# 00 · Environment Setup and Diagnostics

**SIBOL: System for Image-Based Observation and Learning**

Run this notebook first, every time you start a new session (locally or in Google Colab). It:

1. Detects whether you are running **locally** or in **Google Colab**.
2. Mounts **Google Drive** (Colab only) and creates the project folders there.
3. Reports **Python, PyTorch, Torchvision, NumPy and SciPy** versions.
4. Detects **GPU hardware** and **CUDA** availability.
5. Prints a single **diagnostic summary** with any warnings.

> This notebook does **not** reinstall PyTorch or Torchvision. Colab already ships CUDA-enabled builds, and reinstalling them can break the runtime.

## 1. Detect the runtime

In [ ]:
import importlib
import importlib.util
import os
import platform
import shutil
import subprocess
import sys


def running_in_colab() -> bool:
    # True if this notebook is executing inside Google Colab.
    return importlib.util.find_spec("google.colab") is not None


IN_COLAB = running_in_colab()
RUNTIME = "Google Colab" if IN_COLAB else "Local"

print(f"Runtime         : {RUNTIME}")
print(f"Python          : {platform.python_version()} ({sys.executable})")
print(f"Operating system: {platform.system()} {platform.release()}")

## 2. Mount Google Drive (Colab only)

In Colab, files on the VM disappear when the runtime resets, so we keep datasets, checkpoints and outputs on Google Drive under `MyDrive/sibol/`. When running locally this cell does nothing.

In [ ]:
DRIVE_PROJECT_DIR = None

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    DRIVE_PROJECT_DIR = "/content/drive/MyDrive/sibol"
    for sub in ("data", "checkpoints", "outputs/figures", "outputs/metrics"):
        os.makedirs(os.path.join(DRIVE_PROJECT_DIR, sub), exist_ok=True)
    print(f"Google Drive mounted. Project folder: {DRIVE_PROJECT_DIR}")
else:
    print("Not running in Colab: skipping Google Drive mount.")

## 3. Package versions

We read each package's version without failing if it is missing, so you get a full report instead of an error at the first missing package.

In [ ]:
# (display name, import name)
PACKAGES = [
    ("PyTorch", "torch"),
    ("Torchvision", "torchvision"),
    ("NumPy", "numpy"),
    ("SciPy", "scipy"),
    ("scikit-learn", "sklearn"),
    ("scikit-image", "skimage"),
    ("Matplotlib", "matplotlib"),
    ("Pandas", "pandas"),
    ("Pillow", "PIL"),
    ("tqdm", "tqdm"),
    ("Gradio", "gradio"),
]


def get_version(import_name: str):
    # Return the package version string, or None if it is not installed.
    if importlib.util.find_spec(import_name) is None:
        return None
    try:
        module = importlib.import_module(import_name)
        return getattr(module, "__version__", "installed (version unknown)")
    except Exception as exc:  # installed but broken
        return f"import error: {exc}"


VERSIONS = {name: get_version(mod) for name, mod in PACKAGES}
MISSING = [name for name, v in VERSIONS.items() if v is None]

print(f"{'Python':<14}: {platform.python_version()}")
for name, version in VERSIONS.items():
    print(f"{name:<14}: {version or 'NOT INSTALLED'}")

### 3b. Optional: install missing packages

Set `INSTALL_MISSING = True` to install **only** packages that are missing. PyTorch and Torchvision are always excluded: install those yourself from [pytorch.org](https://pytorch.org/get-started/locally/) if you need them locally.

In [ ]:
INSTALL_MISSING = False  # change to True to install missing packages

# Map display names to pip package names (torch/torchvision intentionally excluded).
PIP_NAMES = {
    "NumPy": "numpy",
    "SciPy": "scipy",
    "scikit-learn": "scikit-learn",
    "scikit-image": "scikit-image",
    "Matplotlib": "matplotlib",
    "Pandas": "pandas",
    "Pillow": "pillow",
    "tqdm": "tqdm",
    "Gradio": "gradio",
}

to_install = [PIP_NAMES[name] for name in MISSING if name in PIP_NAMES]

if not to_install:
    print("Nothing to install: all optional packages are present.")
elif not INSTALL_MISSING:
    print("Missing packages:", ", ".join(to_install))
    print("Set INSTALL_MISSING = True and re-run this cell to install them.")
else:
    print("Installing:", ", ".join(to_install))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *to_install], check=True)
    print("Done. Restart the kernel/runtime, then re-run this notebook.")

## 4. GPU and CUDA detection

In [ ]:
TORCH_OK = VERSIONS["PyTorch"] is not None and not str(VERSIONS["PyTorch"]).startswith("import error")
CUDA_AVAILABLE = False
GPU_NAMES = []
DEVICE = "cpu"

if TORCH_OK:
    import torch

    CUDA_AVAILABLE = torch.cuda.is_available()
    print(f"CUDA available      : {CUDA_AVAILABLE}")
    print(f"PyTorch built w/CUDA: {torch.version.cuda or 'no (CPU-only build)'}")

    if CUDA_AVAILABLE:
        DEVICE = "cuda"
        print(f"cuDNN version       : {torch.backends.cudnn.version()}")
        print(f"GPU count           : {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            GPU_NAMES.append(props.name)
            print(
                f"  GPU {i}: {props.name} | "
                f"{props.total_memory / 1024**3:.1f} GB | "
                f"compute capability {props.major}.{props.minor}"
            )
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        DEVICE = "mps"  # Apple Silicon GPU
        print("Apple MPS GPU available (no CUDA).")
    else:
        print("No GPU detected: PyTorch will use the CPU.")
else:
    print("PyTorch is not installed, so GPU detection was skipped.")

# nvidia-smi gives driver-level info (only present on machines with NVIDIA drivers).
if shutil.which("nvidia-smi"):
    print("\n--- nvidia-smi ---")
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 5. Quick device sanity check

A tiny matrix multiplication on the selected device. This confirms the device actually works. It is **not** training or benchmarking.

In [ ]:
if TORCH_OK:
    a = torch.randn(256, 256, device=DEVICE)
    b = torch.randn(256, 256, device=DEVICE)
    c = a @ b
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    print(f"Matrix multiply on '{DEVICE}' succeeded; result shape {tuple(c.shape)}.")
else:
    print("Skipped (PyTorch not installed).")

# PyTorch <-> NumPy interoperability (breaks if torch was built for NumPy 1.x but NumPy 2.x is installed).
TORCH_NUMPY_OK = False
if TORCH_OK and VERSIONS["NumPy"]:
    import numpy as np

    try:
        torch.from_numpy(np.zeros(3)).sum()
        TORCH_NUMPY_OK = True
        print("PyTorch <-> NumPy conversion works.")
    except Exception as exc:
        print(f"PyTorch <-> NumPy conversion FAILED: {exc}")

## 6. Diagnostic summary

In [ ]:
warnings = []
if not TORCH_OK:
    warnings.append("PyTorch is missing: required for the CNN and ResNet-18 experiments.")
if TORCH_OK and VERSIONS["NumPy"] and not TORCH_NUMPY_OK:
    warnings.append("PyTorch cannot exchange data with this NumPy version (often torch built for NumPy 1.x "
                    "with NumPy 2.x installed). Use matching versions, e.g. a fresh virtual environment.")
if VERSIONS["Torchvision"] is None:
    warnings.append("Torchvision is missing: required to load Flowers-102 and ResNet-18.")
if VERSIONS["SciPy"] is None:
    warnings.append("SciPy is missing: required for the classical HOG + SVM pipeline.")
if IN_COLAB and not CUDA_AVAILABLE:
    warnings.append("Colab runtime has no GPU. Use Runtime > Change runtime type > T4 GPU.")
if not IN_COLAB and not CUDA_AVAILABLE:
    warnings.append("No local CUDA GPU: fine for development and the classical model; train CNNs in Colab.")
other_missing = [m for m in MISSING if m not in ("PyTorch", "Torchvision", "SciPy")]
if other_missing:
    warnings.append("Optional packages missing: " + ", ".join(other_missing))

line = "=" * 60
print(line)
print(" SIBOL ENVIRONMENT DIAGNOSTIC")
print(line)
print(f" Runtime       : {RUNTIME}")
print(f" Python        : {platform.python_version()}")
print(f" PyTorch       : {VERSIONS['PyTorch'] or 'NOT INSTALLED'}")
print(f" Torchvision   : {VERSIONS['Torchvision'] or 'NOT INSTALLED'}")
print(f" NumPy         : {VERSIONS['NumPy'] or 'NOT INSTALLED'}")
print(f" SciPy         : {VERSIONS['SciPy'] or 'NOT INSTALLED'}")
print(f" CUDA available: {CUDA_AVAILABLE}")
print(f" GPU(s)        : {', '.join(GPU_NAMES) if GPU_NAMES else 'none'}")
print(f" Torch device  : {DEVICE}")
print(f" Drive folder  : {DRIVE_PROJECT_DIR or 'n/a (local)'}")
print(line)
if warnings:
    print(" WARNINGS:")
    for w in warnings:
        print(f"  - {w}")
else:
    print(" All checks passed.")
print(line)

## Next step

Open `01_dataset_exploration.ipynb` to set up and inspect the Oxford Flowers-102 dataset.